<a href="https://colab.research.google.com/github/deltorobarba/quantum/blob/main/chebyshev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chebyshev Polynomials in Quantum Topological Data Analysis**

https://medium.com/@deltorobarba/chebyshev-polynomials-in-quantum-topological-data-analysis-419c69784a2a

In [2]:
!pip install pennylane -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.0 MB/s eta 0:00:00


In [3]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

In [4]:
# Step 1: Define the quantum walk operator
def quantum_walk_operator():
    coeffs = [1.0, 1.0]
    obs = [qml.PauliX(0) @ qml.PauliX(1), qml.PauliZ(0)]
    return qml.Hamiltonian(coeffs, obs)

# Step 2: Map the spectrum to [-1, 1]
H_original = quantum_walk_operator()
matrix = qml.matrix(H_original)
eigenvalues = np.linalg.eigvalsh(matrix)
a, b = np.min(eigenvalues), np.max(eigenvalues)

def shifted_scaled_hamiltonian():
    shift = -(b + a) / (b - a)
    scale = 2 / (b - a)
    shifted_coeffs = [scale * c for c in H_original.coeffs]
    H_shifted = qml.Hamiltonian(shifted_coeffs, H_original.ops)
    shift_op = shift * qml.Identity(0)
    return H_shifted + shift_op

# Step 3: Define the filter function and compute coefficients
N = 50
def filter_function(x, sigma=0.1):
    return np.exp(-x**2 / (2 * sigma**2))

def compute_chebyshev_coefficients(N):
    k = np.arange(0, N+1)
    c = np.zeros(N+1)
    theta = np.linspace(0, np.pi, 1000)
    x = np.cos(theta)
    w = np.ones_like(theta)
    w[0] /= 2
    w[-1] /= 2
    for i in range(N+1):
        T_k = np.cos(i * theta)
        integrand = filter_function(x) * T_k
        c[i] = (2 - (i == 0)) / np.pi * np.sum(integrand * w) * (theta[1] - theta[0])
    return c

# Step 4: Apply Chebyshev polynomials recursively
def apply_chebyshev_filter(state, H_matrix, c):
    T_prev = state
    T_curr = H_matrix @ state
    filtered_state = c[0] * T_prev + c[1] * T_curr

    for k in range(2, len(c)):
        T_next = 2 * H_matrix @ T_curr - T_prev
        filtered_state += c[k] * T_next
        T_prev = T_curr
        T_curr = T_next

    return filtered_state

# Step 5: Set up the quantum circuit
dev = qml.device('default.qubit', wires=2)
@qml.qnode(dev)
def circuit():
    qml.BasisState(np.array([0, 0]), wires=[0, 1])
    return qml.state()

# Step 6: Run the simulation
state = circuit()
H_shifted = shifted_scaled_hamiltonian()
H_matrix = qml.matrix(H_shifted)
c = compute_chebyshev_coefficients(N)
filtered_state = apply_chebyshev_filter(state, H_matrix, c)
filtered_state /= np.linalg.norm(filtered_state)

# Step 7: Analyze the results
print("Filtered State:")
print(filtered_state)
probabilities = np.abs(filtered_state) ** 2
print("Probabilities:")
print(probabilities)

Filtered State:
[-1.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
  3.60838618e-10+0.j]
Probabilities:
[1.00000000e+00 0.00000000e+00 0.00000000e+00 1.30204508e-19]
